Reranking basique

In [1]:
# Cellule 1 — Installation
!pip install -U pinecone==6.0.1 pinecone-notebooks --quiet
print("Installation terminée !")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.4/421.4 kB 5.6 MB/s eta 0:00:00
Installation terminée !


In [2]:
# Cellule 2 — Authentification Pinecone
import os

PINECONE_API_KEY ='pcsk_6rVesG_6S2LQeKFTGt3evkZ93XPXkZA4cyE6wi4qZ8rpKSiA78wEUdwsrDVmwumrif44rK'

if not os.environ.get("PINECONE_API_KEY"):
    from pinecone_notebooks.colab import Authenticate
    Authenticate()

# Vérification
print("Clé API configurée !")

Clé API configurée !


In [5]:
# Cellule 3 — Initialiser le client Pinecone
from pinecone import Pinecone

PINECONE_API_KEY ='pcsk_6rVesG_6S2LQeKFTGt3evkZ93XPXkZA4cyE6wi4qZ8rpKSiA78wEUdwsrDVmwumrif44rK'

api_key = os.environ.get("PINECONE_API_KEY")
pc      = Pinecone(api_key=PINECONE_API_KEY)

print("Client Pinecone initialisé !")

Client Pinecone initialisé !


In [6]:
# Cellule 4 — Définir la requête et les documents
# On mélange Apple (fruit) et Apple (entreprise) exprès
# pour tester si le reranker comprend le bon contexte

query = "Tell me about Apple's products"

documents = [
    "An apple is a sweet fruit that grows on trees. It comes in many varieties like Gala and Fuji.",
    "Apple Inc. is a technology company that makes the iPhone, MacBook, iPad, and Apple Watch.",
    "Doctors say eating an apple every day can help maintain good health and reduce disease risk.",
    "Apple recently launched the iPhone 15 with improved camera features and faster processing.",
    "The apple tree produces fruit in autumn and requires specific soil conditions to grow well."
]

print(f"Requête : {query}")
print(f"Nombre de documents : {len(documents)}")

Requête : Tell me about Apple's products
Nombre de documents : 5


In [7]:
# Cellule 5 — Appeler le reranker
from pinecone import RerankModel

reranked = pc.inference.rerank(
    model="bge-reranker-v2-m3",   # modèle de reranking
    query=query,
    documents=[
        {"id": str(i), "text": doc}
        for i, doc in enumerate(documents)
    ],
    top_n=3  # retourner les 3 meilleurs résultats
)

print("Reranking terminé !")
print(f"Type de résultat : {type(reranked)}")

Reranking terminé !
Type de résultat : <class 'pinecone.data.features.inference.models.rerank_result.RerankResult'>


In [8]:
# Cellule 6 — Afficher les résultats rerankés
def show_reranked_results(query, matches):
    """
    Affiche les résultats rerankés avec leur score de pertinence.

    Plus le score est élevé, plus le document est pertinent.
    On s'attend à voir les documents sur Apple Inc. en premier.
    """
    print(f"Query: {query}")
    print("-" * 60)
    for i, m in enumerate(matches):
        print(f"\n{i+1}. Score : {m.score:.4f}")
        print(f"   Texte : {m.document.text}")

show_reranked_results(query, reranked.data)

Query: Tell me about Apple's products
------------------------------------------------------------

1. Score : 0.8284
   Texte : Apple Inc. is a technology company that makes the iPhone, MacBook, iPad, and Apple Watch.

2. Score : 0.1285
   Texte : Apple recently launched the iPhone 15 with improved camera features and faster processing.

3. Score : 0.0215
   Texte : An apple is a sweet fruit that grows on trees. It comes in many varieties like Gala and Fuji.


PART 2 — Index Serverless pour Medical Notes

In [9]:
# Cellule 7 — Installation des librairies supplémentaires
!pip install pandas torch transformers --quiet
print("Installation Part 2 terminée !")

Installation Part 2 terminée !


In [10]:
# Cellule 8 — Imports et configuration
import os
import time
import pandas as pd
from pinecone import Pinecone, ServerlessSpec
from transformers import AutoTokenizer, AutoModel
import torch

# Configuration cloud
cloud  = os.getenv('PINECONE_CLOUD',  'aws')
region = os.getenv('PINECONE_REGION', 'us-east-1')

# Spécifications serverless
spec = ServerlessSpec(cloud=cloud, region=region)

# Nom de l'index
index_name = 'medical-notes-index'

print(f"Cloud   : {cloud}")
print(f"Region  : {region}")
print(f"Index   : {index_name}")

Cloud   : aws
Region  : us-east-1
Index   : medical-notes-index


In [11]:
# Cellule 9 — Créer l'index Pinecone
# Supprimer si existe déjà (pour repartir proprement)
if pc.has_index(name=index_name):
    pc.delete_index(name=index_name)
    print(f"Ancien index '{index_name}' supprimé")

# Créer un nouvel index
pc.create_index(
    name=index_name,
    dimension=384,      # dimension du modèle all-MiniLM-L6-v2
    metric='cosine',    # similarité cosinus pour le texte
    spec=spec
)

print(f"Index '{index_name}' créé avec succès !")

Index 'medical-notes-index' créé avec succès !


PART 3 — Charger les données médicales

In [15]:
import pandas as pd

path = "/content/sample_data/sample_notes_data.jsonl"

df = pd.read_json(
    path,
    orient="records",
    lines=True
)

print(f"Data shape : {df.shape}")
print(f"Colonnes : {df.columns.tolist()}")

df.head()

Data shape : (100, 3)
Colonnes : ['id', 'values', 'metadata']


,id,values,metadata
0,P011,"[-0.2027486265, 0.2769146562, -0.1509393603, 0...","{'advice': 'rest, hydrate', 'symptoms': 'heada..."
1,P001,"[0.1842793673, 0.4459365904, -0.0770567134, 0....","{'tests': 'EKG, stress test', 'symptoms': 'che..."
2,P002,"[-0.2040648609, -0.1739618927, -0.2897160649, ...","{'HbA1c': '7.2', 'condition': 'diabetes', 'med..."
3,P003,"[0.1889383644, 0.2924542725, -0.2335938066, -0...","{'symptoms': 'cough, wheezing', 'diagnosis': '..."
4,P004,"[-0.12171068040000001, 0.1674752235, -0.231888...","{'referral': 'dermatology', 'condition': 'susp..."


In [16]:
# Cellule 11 — Inspecter les données
print(f"Nombre de lignes   : {df.shape[0]}")
print(f"Nombre de colonnes : {df.shape[1]}")
print(f"\nPremière ligne :")
print(df.iloc[0])

Nombre de lignes   : 100
Nombre de colonnes : 3

Première ligne :
id                                                       P011
values      [-0.2027486265, 0.2769146562, -0.1509393603, 0...
metadata    {'advice': 'rest, hydrate', 'symptoms': 'heada...
Name: 0, dtype: object


PART 4 — Upsert dans Pinecone

In [17]:
# Cellule 12 — Insérer les données dans l'index
index = pc.Index(name=index_name)

# Upsert depuis le DataFrame
index.upsert_from_dataframe(df)

print("Upsert lancé !")

sending upsert requests:   0%|          | 0/100 [00:00<?, ?it/s]

Upsert lancé !


In [18]:
# Cellule 13 — Attendre que l'index soit prêt
def is_fresh(index):
    """
    Vérifie que les vecteurs sont bien indexés.
    On attend qu'il y en ait au moins 1.
    """
    stats        = index.describe_index_stats()
    vector_count = stats.total_vector_count
    print(f"Vecteurs indexés : {vector_count}")
    return vector_count > 0

print("Attente de l'indexation...")
while not is_fresh(index):
    time.sleep(5)

print("\nIndex prêt !")
index.describe_index_stats()

Attente de l'indexation...
Vecteurs indexés : 100

Index prêt !


{'dimension': 384,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 100}},
 'total_vector_count': 100,
 'vector_type': 'dense'}

PART 5 — Fonction d'embedding et recherche

In [19]:
# Cellule 14 — Fonction d'embedding
def get_embedding(input_question):
    """
    Convertit une question en vecteur d'embedding.

    Utilise all-MiniLM-L6-v2 (384 dimensions)
    — même modèle que celui utilisé pour indexer les données.
    """
    model_name    = 'sentence-transformers/all-MiniLM-L6-v2'
    tokenizer_emb = AutoTokenizer.from_pretrained(model_name)
    model_emb     = AutoModel.from_pretrained(model_name)

    encoded_input = tokenizer_emb(
        input_question,
        padding=True,
        truncation=True,
        return_tensors='pt'
    )

    with torch.no_grad():
        model_output = model_emb(**encoded_input)

    # Moyenne sur la dimension de séquence (dim=1)
    # → transforme (1, seq_len, 384) en (1, 384)
    embedding = model_output.last_hidden_state[0].mean(dim=0)

    return embedding

In [20]:
# Cellule 15 — Recherche sémantique
question = "patient has chest pain and breathing difficulties"

# Générer l'embedding de la question
query = get_embedding(question).tolist()

# Chercher dans Pinecone
results = index.query(
    vector=[query],
    top_k=5,
    include_metadata=True
)

# Trier par score décroissant
sorted_matches = sorted(
    results['matches'],
    key=lambda x: x['score'],
    reverse=True
)

print(f"Résultats pour : '{question}'")
print(f"Nombre de résultats : {len(sorted_matches)}")

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Résultats pour : 'patient has chest pain and breathing difficulties'
Nombre de résultats : 5


PART 6 — Afficher et Reranker les résultats

In [21]:
# Cellule 16 — Afficher les résultats initiaux
def show_results(question, matches):
    """
    Affiche les résultats de la recherche vectorielle initiale.
    Avant le reranking.
    """
    print(f"Question : '{question}'")
    print("\nRésultats (avant reranking) :")
    print("-" * 50)
    for i, match in enumerate(matches):
        print(f"\n{str(i+1).rjust(4)}. ID       : {match['id']}")
        print(f"      Score    : {match['score']:.4f}")
        print(f"      Metadata : {match['metadata']}")

show_results(question, sorted_matches)

Question : 'patient has chest pain and breathing difficulties'

Résultats (avant reranking) :
--------------------------------------------------

   1. ID       : P001
      Score    : 0.6919
      Metadata : {'symptoms': 'chest pain', 'tests': 'EKG, stress test'}

   2. ID       : P016
      Score    : 0.4633
      Metadata : {'condition': 'heart murmur', 'referral': 'cardiology'}

   3. ID       : P003
      Score    : 0.4592
      Metadata : {'diagnosis': 'bronchitis', 'symptoms': 'cough, wheezing', 'treatment': 'antibiotics'}

   4. ID       : P032
      Score    : 0.4404
      Metadata : {'condition': 'asthma', 'treatment': 'nebulizer'}

   5. ID       : P0100
      Score    : 0.4348
      Metadata : {'advice': 'over-the-counter pain relief, stretching', 'symptoms': 'muscle pain'}


In [22]:
# Cellule 17 — Préparer les documents pour le reranking
transformed_documents = [
    {
        'id': match['id'],
        'reranking_field': '; '.join([
            f"{key}: {value}"
            for key, value in match['metadata'].items()
        ])
    }
    for match in results['matches']
]

print("Documents transformés pour reranking :")
for doc in transformed_documents[:2]:
    print(f"\nID : {doc['id']}")
    print(f"Champ : {doc['reranking_field'][:100]}...")

Documents transformés pour reranking :

ID : P001
Champ : symptoms: chest pain; tests: EKG, stress test...

ID : P016
Champ : condition: heart murmur; referral: cardiology...


In [23]:
# Cellule 18 — Reranking des résultats médicaux
# Question plus précise pour le reranking
refined_query = "patient with acute chest pain and shortness of breath, possible cardiac emergency"

reranked_results = pc.inference.rerank(
    model="bge-reranker-v2-m3",
    query=refined_query,
    documents=transformed_documents,
    rank_fields=["reranking_field"],  # champ sur lequel reranker
    top_n=3,                          # garder les 3 meilleurs
    return_documents=True,
)

print(f"Reranking terminé ! {len(reranked_results.data)} résultats")

Reranking terminé ! 3 résultats


In [24]:
# Cellule 19 — Afficher les résultats rerankés
def show_reranked_results(question, matches):
    """
    Affiche les résultats APRÈS reranking.
    Compare avec les résultats initiaux pour voir l'amélioration.
    """
    print(f"Question : '{question}'")
    print("\nRésultats APRÈS reranking :")
    print("-" * 50)
    for i, match in enumerate(matches):
        print(f"\n{str(i+1).rjust(4)}. ID             : {match.document.id}")
        print(f"      Score rerank  : {match.score:.4f}")
        print(f"      Contenu       : {match.document.reranking_field[:100]}...")

show_reranked_results(refined_query, reranked_results.data)

Question : 'patient with acute chest pain and shortness of breath, possible cardiac emergency'

Résultats APRÈS reranking :
--------------------------------------------------

   1. ID             : P001
      Score rerank  : 0.8341
      Contenu       : symptoms: chest pain; tests: EKG, stress test...

   2. ID             : P016
      Score rerank  : 0.0443
      Contenu       : condition: heart murmur; referral: cardiology...

   3. ID             : P003
      Score rerank  : 0.0266
      Contenu       : diagnosis: bronchitis; symptoms: cough, wheezing; treatment: antibiotics...


In [25]:
# Cellule 20 — Nettoyage (supprimer l'index pour éviter les frais)
pc.delete_index(name=index_name)
print(f"Index '{index_name}' supprimé — pas de frais supplémentaires !")

Index 'medical-notes-index' supprimé — pas de frais supplémentaires !
